# Pipeline Tag Prediction
## Cleaning the Untagged Model Cards

**DATASCI 266: Natural Language Processing with Deep Learning**

UC Berkeley, School of Information

---

The original cleaning pass dropped 333,960 rows that had no `pipeline_tag`, since there was no label to train on. This notebook goes back to that dropped pool and cleans it the same way, minus the steps that only make sense when a tag exists (leakage checking on the tag, top-10 tag selection).

Goal: end up with a set of untagged cards that passed the same quality bar as the labeled training set, ready to use later for inference (i.e. predicting tags for real untagged models on the Hub).

Steps:
1. Load raw dataset, isolate untagged rows
2. Strip YAML frontmatter
3. Deduplicate cards
4. Quality / length filtering
5. Compare against the labeled dataset
6. Save

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import hashlib
import warnings
from datasets import load_dataset

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)

print('Libraries loaded check')

Libraries loaded check


## 1. Load Raw Dataset & Isolate Untagged Rows

Same source as the original cleaning pass: `librarian-bots/model_cards_with_metadata`.

This time we keep the rows that have **no** `pipeline_tag`, instead of dropping them. We still require non-empty card text, since a blank card carries no training signal regardless of tag status.

In [ ]:
dataset = load_dataset('librarian-bots/model_cards_with_metadata', split='train')
df_raw = dataset.to_pandas()

print(f'Raw dataset size: {len(df_raw):,} rows')
df_raw.head()

README.md:   0%|          | 0.00/5.89k [00:00<?, ?B/s]

data/train-00000-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  292MB            

data/train-00000-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  297MB            

data/train-00001-of-00004.parquet: downloading bytes:           |  0.00B            

...

data/train-00003-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  294MB            

data/train-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/697461 [00:00<?, ? examples/s]

Raw dataset size: 697,461 rows


,modelId,author,last_modified,downloads,likes,library_name,tags,pipeline_tag,createdAt,card
0,Muapi/dancewithturn-flux,Muapi,2026-05-29 21:27:52+00:00,0,0,diffusers,"[diffusers, lora, text-to-image, stable-diffusion, flux, flux.1-d, base_model:black-forest-labs/FLUX.1-dev, base_mod...",text-to-image,2026-05-29 21:27:17+00:00,---\nlicense: openrail++\nlibrary_name: diffusers\nbase_model: black-forest-labs/FLUX.1-dev\ntags:\n - lora\n - te...
1,Muapi/envy-flux-character-concept-01,Muapi,2026-05-13 20:05:57+00:00,0,0,diffusers,"[diffusers, lora, text-to-image, stable-diffusion, flux, flux.1-d, base_model:black-forest-labs/FLUX.1-dev, base_mod...",text-to-image,2025-08-20 20:41:28+00:00,---\nlicense: openrail++\nlibrary_name: diffusers\nbase_model: black-forest-labs/FLUX.1-dev\ntags:\n - lora\n - te...
2,DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF,DreadPoor,2026-01-24 02:15:12+00:00,10,0,transformers,"[transformers, gguf, merge, mergekit, lazymergekit, llama-cpp, gguf-my-repo, base_model:DreadPoor/Kitsch_Late_ALT-TE...",None,2026-01-24 00:19:09+00:00,---\nlibrary_name: transformers\nlicense: cc-by-nc-4.0\ntags:\n- merge\n- mergekit\n- lazymergekit\n- llama-cpp\n- g...
3,kerenOrr/q-FrozenLake-v1-4x4-noSlippery,kerenOrr,2026-04-28 05:20:48+00:00,0,0,None,"[FrozenLake-v1-4x4-no_slippery, q-learning, reinforcement-learning, custom-implementation, model-index, region:us]",reinforcement-learning,2026-04-28 05:20:41+00:00,---\ntags:\n- FrozenLake-v1-4x4-no_slippery\n- q-learning\n- reinforcement-learning\n- custom-implementation\nmodel-...
4,huuthanh615/model_1375,huuthanh615,2025-12-20 08:40:17+00:00,0,0,None,"[license:creativeml-openrail-m, region:us]",None,2025-12-20 08:40:16+00:00,---\r\nlicense: creativeml-openrail-m\r\n---\r\n


In [ ]:
# Untagged = pipeline_tag is null or empty string
is_missing_tag = df_raw['pipeline_tag'].isna() | (df_raw['pipeline_tag'].astype(str).str.strip() == '')

df_untagged = df_raw[is_missing_tag].copy()
print(f'Untagged rows (no pipeline_tag): {len(df_untagged):,} ({len(df_untagged) / len(df_raw):.1%} of raw)')

# Same missing-card-text drop as the original Step 2
before = len(df_untagged)
df_untagged = df_untagged[df_untagged['card'].notna() & (df_untagged['card'].astype(str).str.strip() != '')].copy()
print(f'Dropped {before - len(df_untagged):,} rows with empty/null card text')
print(f'Remaining: {len(df_untagged):,} rows')

Untagged rows (no pipeline_tag): 341,471 (49.0% of raw)
Dropped 0 rows with empty/null card text
Remaining: 341,471 rows


A quick sanity check: this should land close to the 333,960 figure from the original cleaning doc, since that number already accounted for missing card text among the untagged pool. Small drift is expected since the Hub crawl keeps growing.

## 2. Strip YAML Frontmatter

Untagged cards can still carry a YAML header with other fields (`license`, `tags`, `base_model`, etc), just not `pipeline_tag`. We strip it the same way as the original pipeline: keep the prose, keep the YAML block in a separate column for reference.

No leakage check is needed here since there's no tag value to leak.

In [ ]:
YAML_PATTERN = re.compile(r'^---\s*\n(.*?\n)---\s*\n?', re.DOTALL)

def strip_yaml(text):
    if not isinstance(text, str):
        return text, None
    match = YAML_PATTERN.match(text)
    if match:
        yaml_block = match.group(1)
        prose = text[match.end():]
        return prose, yaml_block
    return text, None

stripped_results = df_untagged['card'].apply(strip_yaml)
df_untagged['text'] = stripped_results.apply(lambda x: x[0])
df_untagged['yaml_raw'] = stripped_results.apply(lambda x: x[1])

n_stripped = df_untagged['yaml_raw'].notna().sum()
print(f'YAML block detected and stripped: {n_stripped:,} rows ({n_stripped / len(df_untagged):.1%})')

YAML block detected and stripped: 308,489 rows (90.3%)


In [ ]:
# Clean up whitespace left over after stripping, same as original pipeline intent
df_untagged['text'] = df_untagged['text'].astype(str).str.strip()
df_untagged = df_untagged[df_untagged['text'] != ''].copy()
print(f'Remaining after removing rows with no prose left: {len(df_untagged):,}')

Remaining after removing rows with no prose left: 327,295


## 3. Deduplicate Cards

Same normalized-text hash approach as the original: lowercase, strip the model's own name, strip digits, collapse whitespace, then hash. Empty-after-normalization rows get dropped outright. Everything else gets deduplicated, keeping one representative per group.

In [ ]:
def normalize_for_hash(text, model_id):
    if not isinstance(text, str):
        return ''
    norm = text.lower()
    if isinstance(model_id, str):
        model_name = model_id.split('/')[-1].lower()
        norm = norm.replace(model_name, '')
    norm = re.sub(r'\d+', '', norm)
    norm = re.sub(r'[^\w\s]', '', norm)
    norm = re.sub(r'\s+', ' ', norm).strip()
    return norm

df_untagged['norm_text'] = df_untagged.apply(
    lambda row: normalize_for_hash(row['text'], row['modelId']), axis=1
)

# Drop rows that normalize to nothing (no real content signal)
before = len(df_untagged)
df_untagged = df_untagged[df_untagged['norm_text'] != ''].copy()
print(f'Dropped {before - len(df_untagged):,} rows normalizing to empty content')

df_untagged['content_hash'] = df_untagged['norm_text'].apply(
    lambda x: hashlib.md5(x.encode('utf-8')).hexdigest()
)

Dropped 148 rows normalizing to empty content


In [ ]:
dup_counts = df_untagged['content_hash'].value_counts()
n_dup_groups = (dup_counts > 1).sum()
n_dup_rows = dup_counts[dup_counts > 1].sum()
print(f'Duplicate groups found: {n_dup_groups:,}, covering {n_dup_rows:,} rows')

before = len(df_untagged)
df_untagged = df_untagged.drop_duplicates(subset='content_hash', keep='first').copy()
print(f'Dropped {before - len(df_untagged):,} duplicate rows, kept one representative per group')
print(f'Remaining: {len(df_untagged):,} rows')

Duplicate groups found: 12,558, covering 221,659 rows
Dropped 209,101 duplicate rows, kept one representative per group
Remaining: 118,046 rows


Spot check: same two known-template hashes from the original doc (the default `transformers` push_to_hub template) should show up here too, since untagged models are just as likely to be unfilled templates.

In [ ]:
sample_dupe_hashes = dup_counts[dup_counts > 1].index[:3]
for h in sample_dupe_hashes:
    example = df_untagged[df_untagged['content_hash'] == h]
    if len(example) > 0:
        print(example['text'].iloc[0][:200])
        print('---')

# Gensyn BlockAssist

Gensyn's BlockAssist is a distributed extension of the paper [AssistanceZero: Scalably Solving Assistance Games](https://arxiv.org/abs/2504.07091).
---
# Model Card for Model ID

<!-- Provide a quick summary of what the model is/does. -->



## Model Details

### Model Description

<!-- Provide a longer summary of what this model is. -->

This is the
---
[View on Civ Archive](https://civarchive.com/models/1202133?modelVersionId=1353640)
---


## 4. Quality / Length Filtering

Identical thresholds to the original: character length between 100 and 100,000, minimum unique-word ratio of 0.15. Using the same numbers here is deliberate, the point of this dataset is to be quality-matched to the labeled training set, not independently tuned.

In [ ]:
MIN_CHARS = 100
MAX_CHARS = 100_000
MIN_UNIQUE_WORD_RATIO = 0.15

df_untagged['char_len'] = df_untagged['text'].str.len()

def unique_word_ratio(text):
    words = text.split()
    if len(words) == 0:
        return 0.0
    return len(set(words)) / len(words)

df_untagged['word_count'] = df_untagged['text'].str.split().str.len()
df_untagged['unique_word_ratio'] = df_untagged['text'].apply(unique_word_ratio)

before = len(df_untagged)
mask = (
    (df_untagged['char_len'] >= MIN_CHARS) &
    (df_untagged['char_len'] <= MAX_CHARS) &
    (df_untagged['unique_word_ratio'] >= MIN_UNIQUE_WORD_RATIO)
)
df_untagged = df_untagged[mask].copy()
print(f'Dropped {before - len(df_untagged):,} rows failing quality/length filters')
print(f'Remaining: {len(df_untagged):,} rows')

Dropped 2,426 rows failing quality/length filters
Remaining: 115,620 rows


In [ ]:
df_untagged['char_len'].describe()

,char_len
count,115620.000000
mean,3033.794231
std,4246.082861
min,100.000000
25%,834.000000
50%,1794.000000
75%,3723.000000
max,95473.000000


## 5. Compare Against the Labeled Dataset

Loading the cleaned labeled dataset back in to check the two sets look comparable on length and quality, not just matched by threshold. If the untagged set skews noticeably shorter or lower quality even after filtering, that's worth knowing before using it for anything downstream (e.g. inference or semi-supervised extensions).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/266-pipeline-tag-prediction'

try:
    df_labeled = pd.read_parquet(f'{DRIVE_DIR}/model_cards_cleaned.parquet')
    comparison = pd.DataFrame({
        'labeled': df_labeled['char_len'].describe(),
        'untagged': df_untagged['char_len'].describe()
    })
    print(comparison)
except FileNotFoundError:
    print('Labeled dataset not found at that path, skipping comparison. Check DRIVE_DIR.')

Mounted at /content/drive
             labeled       untagged
count  141036.000000  115620.000000
mean     4364.977417    3033.794231
std      7900.609954    4246.082861
min       100.000000     100.000000
25%      1073.000000     834.000000
50%      2039.500000    1794.000000
75%      4514.000000    3723.000000
max     99615.000000   95473.000000


## 6. Final Summary & Save

Dropping the helper columns used only for cleaning (`norm_text`, `content_hash`, `unique_word_ratio`, `yaml_raw`) before saving, same column discipline as the labeled dataset. Kept `yaml_raw` out of the final save since it's large and was only needed for QA, same call the original pipeline made.

In [ ]:
print('Untagged cleaning summary')
print('-' * 40)
print(f'Final row count: {len(df_untagged):,}')
print(f'Columns: {list(df_untagged.columns)}')

Untagged cleaning summary
----------------------------------------
Final row count: 115,620
Columns: ['modelId', 'author', 'last_modified', 'downloads', 'likes', 'library_name', 'tags', 'pipeline_tag', 'createdAt', 'card', 'text', 'yaml_raw', 'norm_text', 'content_hash', 'char_len', 'word_count', 'unique_word_ratio']


In [ ]:
final_cols = ['modelId', 'text', 'char_len', 'word_count']
df_final = df_untagged[final_cols].copy()

df_final.to_parquet(f'{DRIVE_DIR}/model_cards_untagged_cleaned.parquet', index=False)
print(f'Saved {len(df_final):,} rows to {DRIVE_DIR}/model_cards_untagged_cleaned.parquet')

Saved 115,620 rows to /content/drive/MyDrive/266-pipeline-tag-prediction/model_cards_untagged_cleaned.parquet


To load this dataset in another notebook:

```python
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/266-pipeline-tag-prediction'
df_untagged = pd.read_parquet(f'{DRIVE_DIR}/model_cards_untagged_cleaned.parquet')
```

### Notes for later use

- This set has no ground-truth label, so it's meant for inference or downstream analysis, not for training or evaluation on its own.
- Quality thresholds are copied exactly from the labeled pipeline (100-100,000 chars, 0.15 unique-word ratio), so any quality difference between the two sets reflects the actual data, not different filtering.
- The two big known-template duplicate groups from the labeled cleaning process likely show up here too, worth a quick spot check if the final count looks unexpectedly large.